In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch.nn.utils.rnn import (
    pad_sequence
)

In [ ]:
TRAIN_FEATURES_DIR = "/content/drive/MyDrive/preprocessed/new_features/train"

CSV_PATH = "/content/drive/MyDrive/preprocessed/oasis_tlstm_ready_cleaned.csv"

MODEL_SAVE_PATH = "/content/drive/MyDrive/preprocessed/tlstm_model.pth"

PCA_SAVE_PATH = "/content/drive/MyDrive/preprocessed/pca_components.npy"

PCA_MEAN_PATH = "/content/drive/MyDrive/preprocessed/pca_mean.npy"

SCALER_MEAN_PATH = "/content/drive/MyDrive/preprocessed/scaler_mean.npy"

SCALER_SCALE_PATH = "/content/drive/MyDrive/preprocessed/scaler_scale.npy"

PCA_COMPONENTS = 128

BATCH_SIZE = 8

EPOCHS = 100

LR = 5e-5

MAX_SEQ_LEN = 3

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
CLINICAL_COLS = [
    'MMSE',
    'eTIV',
    'nWBV',
    'ASF'
]

STATIC_COLS = [
    'EDUC',
    'SES',
    'M/F_M'
]

FEATURE_COLS = (
    CLINICAL_COLS +
    STATIC_COLS
)

TARGET_COL = 'CDR'

PATIENT_COL = 'patient_id'

VISIT_COL = 'Visit'

MRI_COL = 'mri_id'

In [ ]:
df = pd.read_csv(CSV_PATH)

df.head()

,patient_id,mri_id,Group,Visit,mr_delay_raw,M/F,Hand,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF,M/F_M
0,OAS2_0001,OAS2_0001_MR1,Nondemented,1,0,M,R,87,14,2.0,27.0,0.0,1986.550000,0.696106,0.883440,1
1,OAS2_0001,OAS2_0001_MR2,Nondemented,2,457,M,R,88,14,2.0,30.0,0.0,2004.479526,0.681062,0.875539,1
2,OAS2_0002,OAS2_0002_MR1,Demented,1,0,M,R,75,12,2.0,23.0,0.5,1678.290000,0.736336,1.045710,1
3,OAS2_0002,OAS2_0002_MR2,Demented,2,560,M,R,76,12,2.0,28.0,0.5,1737.620000,0.713402,1.010000,1
4,OAS2_0002,OAS2_0002_MR3,Demented,3,1895,M,R,80,12,2.0,22.0,0.5,1697.911134,0.701236,1.033623,1


In [ ]:
scaler = StandardScaler()

df[FEATURE_COLS] = scaler.fit_transform(
    df[FEATURE_COLS]
)

# SAVE SCALER PARAMETERS
np.save(
    SCALER_MEAN_PATH,
    scaler.mean_
)

np.save(
    SCALER_SCALE_PATH,
    scaler.scale_
)

In [ ]:
mri_feature_map = {}

for root, dirs, files in os.walk(
        TRAIN_FEATURES_DIR):

    for file in files:

        if file.endswith(".npy"):

            image_id = file.replace(
                ".npy",
                ""
            )

            feat = np.load(
                os.path.join(root, file)
            )

            mri_feature_map[
                image_id
            ] = feat

print(
    "Loaded features:",
    len(mri_feature_map)
)

Loaded features: 298


In [ ]:
all_features = np.stack(
    list(mri_feature_map.values())
)

pca = PCA(
    n_components=PCA_COMPONENTS
)

all_features_pca = pca.fit_transform(
    all_features
)

# SAVE PCA
import joblib

# SAVE ENTIRE PCA OBJECT
joblib.dump(
    pca,
    "/content/drive/MyDrive/pca.pkl"
)

print("PCA object saved")

print(
    "Reduced shape:",
    all_features_pca.shape
)

PCA object saved
Reduced shape: (298, 128)


In [ ]:
image_ids = list(
    mri_feature_map.keys()
)

mri_feature_map_pca = {}

for i, img_id in enumerate(image_ids):

    mri_feature_map_pca[
        img_id
    ] = all_features_pca[i]

In [ ]:
all_sequences = []

df = df.sort_values(
    [PATIENT_COL, VISIT_COL]
)

grouped = df.groupby(PATIENT_COL)

for patient_id, patient_df in grouped:

    clinical_sequence = []

    mri_sequence = []

    patient_df = patient_df.sort_values(
        VISIT_COL
    )

    for _, row in patient_df.iterrows():

        image_id = row[MRI_COL]

        if image_id not in mri_feature_map_pca:
            continue

        clinical_feat = row[
            FEATURE_COLS
        ].values.astype(np.float32)

        mri_feat = mri_feature_map_pca[
            image_id
        ].astype(np.float32)

        clinical_sequence.append(
            clinical_feat
        )

        mri_sequence.append(
            mri_feat
        )

    if len(clinical_sequence) < 2:
        continue

    clinical_sequence = (
        clinical_sequence[-MAX_SEQ_LEN:]
    )

    mri_sequence = (
        mri_sequence[-MAX_SEQ_LEN:]
    )

    target = np.float32(
        patient_df.iloc[-1][TARGET_COL]
    )

    all_sequences.append(
        (
            np.array(clinical_sequence),
            np.array(mri_sequence),
            target
        )
    )

print(
    "Sequences:",
    len(all_sequences)
)

Sequences: 120


In [ ]:
train_seqs = all_sequences

In [ ]:
class ProgressionDataset(Dataset):

    def __init__(self, sequences):

        self.sequences = sequences

    def __len__(self):

        return len(self.sequences)

    def __getitem__(self, idx):

        clinical_x, mri_x, y = (
            self.sequences[idx]
        )

        return (
            torch.tensor(
                clinical_x,
                dtype=torch.float32
            ),

            torch.tensor(
                mri_x,
                dtype=torch.float32
            ),

            torch.tensor(
                y,
                dtype=torch.float32
            )
        )

In [ ]:
def collate_fn(batch):

    clinical_x = [
        item[0] for item in batch
    ]

    mri_x = [
        item[1] for item in batch
    ]

    y = [
        item[2] for item in batch
    ]

    clinical_x = pad_sequence(
        clinical_x,
        batch_first=True,
        padding_value=0
    )

    mri_x = pad_sequence(
        mri_x,
        batch_first=True,
        padding_value=0
    )

    y = torch.tensor(
        y,
        dtype=torch.float32
    )

    return clinical_x, mri_x, y

In [ ]:
train_dataset = ProgressionDataset(
    train_seqs
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
class TLSTM(nn.Module):

    def __init__(self,
                 clinical_dim,
                 mri_dim,
                 hidden_dim=64,
                 dropout=0.3):

        super(TLSTM, self).__init__()

        input_dim = (
            clinical_dim + mri_dim
        )

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.regressor = nn.Sequential(

            nn.Linear(hidden_dim, 32),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(32, 1)
        )

    def forward(self,
                clinical_x,
                mri_x):

        x = torch.cat(
            [clinical_x, mri_x],
            dim=-1
        )

        lstm_out, _ = self.lstm(x)

        out = self.regressor(
            lstm_out[:, -1, :]
        )

        return out

In [ ]:
model = TLSTM(
    clinical_dim=len(FEATURE_COLS),
    mri_dim=PCA_COMPONENTS
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

In [ ]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for clinical_x, mri_x, y in train_loader:

        clinical_x = clinical_x.to(device)

        mri_x = mri_x.to(device)

        y = y.to(device)

        optimizer.zero_grad()

        preds = model(
            clinical_x,
            mri_x
        ).squeeze()

        loss = criterion(
            preds,
            y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (
        total_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {avg_loss:.4f}"
    )

Epoch 1/100 | Loss: 0.3333
Epoch 2/100 | Loss: 0.3126
Epoch 3/100 | Loss: 0.2961
Epoch 4/100 | Loss: 0.2900
Epoch 5/100 | Loss: 0.2786
Epoch 6/100 | Loss: 0.2657
Epoch 7/100 | Loss: 0.2597
Epoch 8/100 | Loss: 0.2578
Epoch 9/100 | Loss: 0.2401
Epoch 10/100 | Loss: 0.2368
Epoch 11/100 | Loss: 0.2222
Epoch 12/100 | Loss: 0.2190
Epoch 13/100 | Loss: 0.2093
Epoch 14/100 | Loss: 0.2023
Epoch 15/100 | Loss: 0.2024
Epoch 16/100 | Loss: 0.1895
Epoch 17/100 | Loss: 0.1819
Epoch 18/100 | Loss: 0.1775
Epoch 19/100 | Loss: 0.1622
Epoch 20/100 | Loss: 0.1532
Epoch 21/100 | Loss: 0.1475
Epoch 22/100 | Loss: 0.1542
Epoch 23/100 | Loss: 0.1370
Epoch 24/100 | Loss: 0.1381
Epoch 25/100 | Loss: 0.1223
Epoch 26/100 | Loss: 0.1268
Epoch 27/100 | Loss: 0.1208
Epoch 28/100 | Loss: 0.1115
Epoch 29/100 | Loss: 0.1056
Epoch 30/100 | Loss: 0.1062
Epoch 31/100 | Loss: 0.1021
Epoch 32/100 | Loss: 0.1041
Epoch 33/100 | Loss: 0.0983
Epoch 34/100 | Loss: 0.0919
Epoch 35/100 | Loss: 0.0860
Epoch 36/100 | Loss: 0.0928
E

In [ ]:
torch.save(
    model.state_dict(),
    MODEL_SAVE_PATH
)

print("Model saved")

Model saved
